### Increase exposure time to make a ramp

Ramps are essential for characterization. Make sure it works!

In [23]:
# CAMERA SERVER
from camera_control.client import CameraClient

# Connect to the camera server
cam = CameraClient("localhost", 5555)
cam.connect()

# Make sure the camera has all corrections turned off
cam.set_correction("GAIN", "OFF")
cam.set_correction("OFFSET", "OFF")   
cam.set_correction("SUB", "OFF")
cam.set_observer_name("Nate Lourie")
cam.set_object_name("PTC Ramp")

Connected to camera server at localhost:5555


{'status': 'success', 'message': 'Object set to PTC Ramp'}

In [24]:
# LAB HARDWARE CONTROL SERVER
import numpy as numpy
import time
import os
import matplotlib.pyplot as plt
# Lab Hardware Control
import Pyro5.api as pyro
import json

# Connect to the lab server
labserver = pyro.Proxy("PYRO:LabServer@localhost:50000")

In [32]:
# Set up the exposure times
import numpy as np
import os


# Base directory
base_dir = os.path.join(os.path.expanduser("~"), "data", "ramps")

# Date-based prefix
date_prefix = time.strftime("%Y%m%d")

# make a list of exposure times:
# n points, log spaced between 0.0001 and 5 seconds
exptimes = np.exp(np.linspace(np.log(0.0001), np.log(5), num=50))

# the exposure times should be rounded to 4 decimal places
exptimes = np.round(exptimes, 4)

exptimes = [ #0.01, 0.02, 0.025, 0.05, 0.075, 0.1,
                0.2,
             #0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 
             #2.0, 2.5, 3.0, 3.5, 
             4.0, 4.5, 5.0,
             ]

# delete any repetitions in the exposure times
exptimes = np.unique(exptimes)

print(f"{len(exptimes)} exposure times: {exptimes}")

4 exposure times: [0.2 4.  4.5 5. ]


In [33]:
# DO THE RAMP
print(f"Exposure times: {exptimes}")
nframes = 10  # Number of frames to capture for each exposure time

# Set up the filter
filter = 1050
labserver.select_bandpass(filter)
wheels = labserver.list_wheels()

# settle time after running shutter
settle_time_s = 2.0  # seconds

for i, exptime in enumerate(exptimes):
    print(f"exposure time [{i+1}/{len(exptimes)}]: {exptime:.4f} seconds")
    print(f"Setting exposure time to {exptime} seconds")
    cam.set_exposure(exptime)
    
    save_dir = os.path.join(base_dir, f"{date_prefix}_{filter}", f"exptime_{exptime:.4f}")
    #-------------- DARKS --------------
    dark_dir = os.path.join(save_dir, "darks")
    if not os.path.exists(dark_dir):
        os.makedirs(dark_dir)
    cam.set_save_path(dark_dir)

    # First take a dark frame
    labserver.shutter("close")
    shutter_status = labserver.status()["shutter"]["state"]
    # wait for the system to settle
    print(f"\tWaiting for {settle_time_s} seconds for the system to settle...")
    time.sleep(settle_time_s)

    header = [
        ("FILTER", filter, "Filter band center in nm"),
        ("GEPDCURR", "", "Ge photodiode current in A"),
        ("SIPDCURR", "", "Si photodiode current in A"),
        ("GEPDSTD", "", "Ge photodiode current standard deviation in A"),
        ("SIPDSTD", "", "Si photodiode current standard deviation in A"),
        ("GEPDMEAN", "", "Ge photodiode current mean in A"),
        ("SIPDMEAN", "", "Si photodiode current mean in A"),
        ("REXPTIME", exptime, "Requested exposure time in seconds"),
        ("COMMENT", "Dark image for PTC Ramp measurement", ""),
        ("SHUTTER", shutter_status, "Shutter status: open or closed"),
        ("FW1FILT", labserver.wheel_status("fw1")['current_filter'], "Current filter in fw1"),
        ("FW2FILT", labserver.wheel_status("fw2")['current_filter'], "Current filter in fw2"),
        ("FW3FILT", labserver.wheel_status("fw3")['current_filter'], "Current filter in fw3"),
        ("FW4FILT", labserver.wheel_status("fw4")['current_filter'], "Current filter in fw4"),
    ]
    cam.capture_frames(nframes, stack=True, headers=header, 
                       show_progress=False)

    #---------------- LIGHTS --------------
    light_dir = os.path.join(save_dir, "lights")
    if not os.path.exists(light_dir):
        os.makedirs(light_dir)
    cam.set_save_path(light_dir)
    # Now take a light frame
    labserver.shutter("open")
    shutter_status = labserver.status()["shutter"]["state"]
    # wait for the system to settle
    print(f"\tWaiting for {settle_time_s} seconds for the system to settle...")
    time.sleep(settle_time_s)

    header = [
        ("FILTER", filter, "Filter band center in nm"),
        ("GEPDCURR", "", "Ge photodiode current in A"),
        ("SIPDCURR", "", "Si photodiode current in A"),
        ("GEPDSTD", "", "Ge photodiode current standard deviation in A"),
        ("SIPDSTD", "", "Si photodiode current standard deviation in A"),
        ("GEPDMEAN", "", "Ge photodiode current mean in A"),
        ("SIPDMEAN", "", "Si photodiode current mean in A"),
        ("REXPTIME", exptime, "Requested exposure time in seconds"),
        ("COMMENT", "Dark image for PTC Ramp measurement", ""),
        ("SHUTTER", shutter_status, "Shutter status: open or closed"),
        ("FW1FILT", labserver.wheel_status("fw1")['current_filter'], "Current filter in fw1"),
        ("FW2FILT", labserver.wheel_status("fw2")['current_filter'], "Current filter in fw2"),
        ("FW3FILT", labserver.wheel_status("fw3")['current_filter'], "Current filter in fw3"),
        ("FW4FILT", labserver.wheel_status("fw4")['current_filter'], "Current filter in fw4"),
    ]
    print("capturing light frames")
    cam.capture_frames(nframes, stack=True, headers=header,
                       show_progress=False)

    

Exposure times: [0.2 4.  4.5 5. ]
exposure time [1/4]: 0.2000 seconds
Setting exposure time to 0.2 seconds
	Waiting for 2.0 seconds for the system to settle...
	Waiting for 2.0 seconds for the system to settle...
capturing light frames
exposure time [2/4]: 4.0000 seconds
Setting exposure time to 4.0 seconds
	Waiting for 2.0 seconds for the system to settle...
	Waiting for 2.0 seconds for the system to settle...
capturing light frames
exposure time [3/4]: 4.5000 seconds
Setting exposure time to 4.5 seconds
	Waiting for 2.0 seconds for the system to settle...
	Waiting for 2.0 seconds for the system to settle...
capturing light frames
exposure time [4/4]: 5.0000 seconds
Setting exposure time to 5.0 seconds
	Waiting for 2.0 seconds for the system to settle...
	Waiting for 2.0 seconds for the system to settle...
capturing light frames
